In [20]:
import pandas as pd

df = pd.read_csv('data/merged_30min_v2.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert('Australia/Sydney')
df['episode_day'] = (df['timestamp'] - pd.Timedelta(minutes=1)).dt.date
steps = df.groupby(['customer_id', 'episode_day']).size()
incomplete_days = steps[steps != 48]
for customer in df['customer_id'].unique():
    n = (incomplete_days.index.get_level_values('customer_id') == customer).sum()
    print(f'Customer {customer} has {n} days with steps != 48')
day_per_customer = df.groupby('customer_id')['episode_day'].nunique()
print(day_per_customer)

Customer 1 has 2 days with steps != 48
Customer 2 has 2 days with steps != 48
Customer 3 has 2 days with steps != 48
Customer 4 has 2 days with steps != 48
Customer 5 has 2 days with steps != 48
customer_id
1    365
2    284
3    365
4    365
5    365
Name: episode_day, dtype: int64


In [21]:
df = df[df['customer_id'] != 2].copy()

steps = df.groupby(['customer_id', 'episode_day'])['timestamp'].transform('size')
df = df[steps == 48].copy()

In [ ]:
import numpy as np

def split_data(dias, seed=50, fr_train=0.70, fr_val=0.15):
    rng = np.random.default_rng(seed=seed)
    d = pd.DataFrame({'day': pd.Series(sorted(dias))})
    d['month'] = pd.to_datetime(d['day']).dt.strftime('%Y-%m')
    train, val, test = [], [], []
    for _, g in d.groupby('month'):     
        arr = g['day'].to_numpy()
        arr = arr[rng.permutation(len(arr))]
        total_size = len(arr)
        train_size = int(round(fr_train * total_size))
        val_size = int(round(fr_val * total_size))
        train += list(arr[:train_size])   
        val   += list(arr[train_size:train_size+val_size])
        test  += list(arr[train_size+val_size:])
    return train, val, test

In [24]:
train_days, val_days, test_days = split_data(df['episode_day'].unique())

total = df['episode_day'].nunique()

print("ok:", len(train_days), len(val_days), len(test_days), "total: ", total)

ok: 256 54 53 total:  363
